# 07 — Seed-Robust Strategy Evaluation for All Accounts

## Why this notebook exists

Every backtest comparison before this one used `random_seed=0` for CatBoost. The seed-variance test (`scripts/seed_variance_test.py`) showed that for **a single fixed cutoff (2024-01-01 corrupted)**, bankroll across 10 seeds spans **$244k → $9.0M** — a 36× spread from seed alone. That is wider than the bankroll spread we observed across 8 *different* cutoffs at seed=0 ($11k → $3.1M).

Consequence: most of our "rolling vs frozen" and "which cutoff" claims are contaminated by seed-level luck. In particular, the "frozen 2024-01-01 is best" claim was an artifact of seed=0 — the 2025-01-01 median across seeds ($2.87M) actually beats the 2024-01-01 median ($1.45M).

**Goal of this notebook:** make the deployment decision on seed-robust evidence. For each of the three accounts (A, B, C) and each of the 8 candidate strategies, we report the distribution of bankrolls across 10 seeds and run within-seed paired McNemar / bootstrap tests.

**Data source:** `data/interim/all_accounts_seed_grid_bets.parquet` and `artifacts/metrics/all_accounts_seed_grid.json`, produced by `scripts/all_accounts_seed_grid.py`.

**Grid:**
- Accounts: A (real, 10%-K + 10% cap), B (real, ¼-K no cap), C (corrupt, ¼-K no cap)
- Strategies (8): frozen 2023-01 / 2024-01 / 2025-01 / 2025-07; rolling 1mo / 3mo / 6mo / 12mo
- Seeds: 0–9
- Eval window: 2025-07-01 → 2026-05-31 (435 Polymarket-matched fights, 11 months)

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from ufc_pred.backtest.strategy_grid import mcnemar_exact, boot_ci, STRATEGIES

summary = pd.DataFrame(json.loads((ROOT / 'artifacts/metrics/all_accounts_seed_grid.json').read_text())['summary'])
bets    = pd.read_parquet(ROOT / 'data/interim/all_accounts_seed_grid_bets.parquet')

print(f'Summary rows : {len(summary)}  (accounts × strategies × seeds = {summary.account.nunique()} × {summary.strategy.nunique()} × {summary.seed.nunique()})')
print(f'Bet rows     : {len(bets):,}')
print(f'Seeds present: {sorted(summary.seed.unique())}')
summary.head()

## 1. Bankroll distribution per (account, strategy)

Each box = 10 seeds. Y-axis is log because winners and losers differ by 1-2 orders of magnitude even at the median level.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
for ax, acct in zip(axes, ['A', 'B', 'C']):
    data = [summary[(summary.account==acct) & (summary.strategy==s)].final.values for s in STRATEGIES]
    ax.boxplot(data, labels=STRATEGIES, showfliers=True)
    ax.set_yscale('log')
    ax.set_ylabel(f'Account {acct}  final bankroll  (log $)')
    ax.axhline(300, color='gray', linestyle=':', linewidth=1, label='start ($300)')
    ax.grid(True, which='both', alpha=0.3)
axes[-1].set_xlabel('Strategy')
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.suptitle('Final bankroll across 10 seeds — per account × strategy', y=1.00)
fig.tight_layout()
plt.show()

## 2. Bankroll summary table

Best median per account is **bolded** in the rendered table.

In [ ]:
def fmt_row(g):
    return pd.Series({
        'min':    g.final.min(),
        'p25':    g.final.quantile(0.25),
        'median': g.final.median(),
        'p75':    g.final.quantile(0.75),
        'max':    g.final.max(),
        'ratio':  g.final.max() / g.final.min(),
        'n_bets_med': int(g.n_bets.median()),
        'hit_med':    g.hit.median(),
    })

tab = summary.groupby(['account', 'strategy']).apply(fmt_row).reset_index()
tab = tab.sort_values(['account', 'median'], ascending=[True, False]).reset_index(drop=True)
tab['is_best'] = tab['median'] == tab.groupby('account')['median'].transform('max')

def hilite(df):
    sty = df.drop(columns=['is_best']).style.format({
        'min':'${:,.0f}', 'p25':'${:,.0f}', 'median':'${:,.0f}',
        'p75':'${:,.0f}', 'max':'${:,.0f}', 'ratio':'{:.1f}x', 'hit_med':'{:.3f}',
    })
    def row_style(r):
        bold = df.loc[r.name, 'is_best']
        return ['font-weight: bold' if bold else '' for _ in r]
    return sty.apply(row_style, axis=1)

hilite(tab)


## 3. Hit-rate distribution per (account, strategy)

Lets us check whether the median-bankroll winner also wins on raw hit rate, or whether the bankroll lead is purely compounding noise.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
for ax, acct in zip(axes, ['A', 'B', 'C']):
    data = [summary[(summary.account==acct) & (summary.strategy==s)].hit.values for s in STRATEGIES]
    ax.boxplot(data, labels=STRATEGIES, showfliers=True)
    ax.set_ylabel(f'Account {acct}  hit rate')
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Strategy')
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.suptitle('Hit rate across 10 seeds — per account × strategy', y=1.00)
fig.tight_layout()
plt.show()

## 4. Within-seed paired comparisons

For each account and each strategy pair (A, B), within each seed we compute:
- McNemar p on side-pick wins paired on the same fight
- Bootstrap mean and 95% CI on per-bet flat-PnL difference

Then we aggregate across seeds by reporting the **median** of each statistic. A median p < 0.05 means "in most seeds, the difference reached significance". A median PnL-diff CI that does not cross zero is strong evidence the per-bet edge difference is real (not just compounding luck).

In [ ]:
def pairwise_within_seed(bets, account):
    sub = bets[bets.account == account]
    rows = []
    for seed in sorted(sub.seed.unique()):
        ss = sub[sub.seed == seed]
        per_strat = {s: ss[ss.strategy == s].set_index('fight_id')[['won','pnl_flat']] for s in STRATEGIES}
        for i in range(len(STRATEGIES)):
            for j in range(i+1, len(STRATEGIES)):
                A, B = STRATEGIES[i], STRATEGIES[j]
                a, b = per_strat[A], per_strat[B]
                common = a.index.intersection(b.index)
                if len(common) == 0: continue
                a2 = a.loc[common][~a.loc[common].index.duplicated()]
                b2 = b.loc[common][~b.loc[common].index.duplicated()]
                common = a2.index.intersection(b2.index)
                a2 = a2.loc[common]; b2 = b2.loc[common]
                bw = int(((a2.won == 1) & (b2.won == 0)).sum())
                cw = int(((a2.won == 0) & (b2.won == 1)).sum())
                p = mcnemar_exact(bw, cw)
                diff = (a2.pnl_flat.values - b2.pnl_flat.values)
                mean, lo, hi = boot_ci(diff, n=2000, seed=seed)
                rows.append({'seed':seed,'A':A,'B':B,'paired_n':len(common),
                             'mcp':p,'pnl_diff':mean,'ci_lo':lo,'ci_hi':hi,
                             'hit_A':float(a2.won.mean()),'hit_B':float(b2.won.mean())})
    return pd.DataFrame(rows)

pair_dfs = {acct: pairwise_within_seed(bets, acct) for acct in ['A', 'B', 'C']}
print({k: len(v) for k, v in pair_dfs.items()})
pair_dfs['C'].head()

In [ ]:
def agg_pair(df):
    return df.groupby(['A', 'B']).agg(
        median_mcp=('mcp', 'median'),
        median_pnl_diff=('pnl_diff', 'median'),
        median_ci_lo=('ci_lo', 'median'),
        median_ci_hi=('ci_hi', 'median'),
        median_hit_A=('hit_A', 'median'),
        median_hit_B=('hit_B', 'median'),
    ).reset_index()

for acct in ['A', 'B', 'C']:
    print(f'\n=== Account {acct} — median across seeds ===')
    a = agg_pair(pair_dfs[acct]).sort_values('median_pnl_diff', ascending=False)
    a['sig'] = (a['median_mcp'] < 0.05) & ~((a['median_ci_lo'] <= 0) & (0 <= a['median_ci_hi']))
    display(a.style.format({
        'median_mcp':'{:.3f}', 'median_pnl_diff':'{:+.3f}',
        'median_ci_lo':'{:+.3f}', 'median_ci_hi':'{:+.3f}',
        'median_hit_A':'{:.3f}', 'median_hit_B':'{:.3f}',
    }))

### 4b. Heatmap: median PnL-diff between strategies, per account

Row strategy `A` vs column strategy `B`. Value = median across seeds of (per-bet flat PnL of A) − (per-bet flat PnL of B). Positive (red) = A is better than B; negative (blue) = A is worse.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, acct in zip(axes, ['A', 'B', 'C']):
    a = agg_pair(pair_dfs[acct])
    M = pd.DataFrame(np.nan, index=STRATEGIES, columns=STRATEGIES, dtype=float)
    for _, r in a.iterrows():
        M.loc[r.A, r.B] =  r.median_pnl_diff
        M.loc[r.B, r.A] = -r.median_pnl_diff
    im = ax.imshow(M.values, cmap='RdBu_r', vmin=-0.1, vmax=0.1, aspect='auto')
    ax.set_xticks(range(len(STRATEGIES)))
    ax.set_yticks(range(len(STRATEGIES)))
    ax.set_xticklabels(STRATEGIES, rotation=45, ha='right')
    ax.set_yticklabels(STRATEGIES)
    ax.set_title(f'Account {acct}: median(PnL_row − PnL_col)')
    for i in range(len(STRATEGIES)):
        for j in range(len(STRATEGIES)):
            v = M.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=8,
                        color='white' if abs(v) > 0.05 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout()
plt.show()

## 5. Cross-seed bootstrap CI on the median bankroll

For each (account, strategy), resample the 10 seeds with replacement (5000×) and report 95% CI on the median bankroll. This is the headline robustness figure — if a strategy's median bankroll CI overlaps another strategy's CI, we cannot reliably say one is better than the other.

In [ ]:
def boot_median_ci(values, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(n, len(values)))
    medians = np.median(values[idx], axis=1)
    return float(np.median(values)), float(np.percentile(medians, 2.5)), float(np.percentile(medians, 97.5))

rows = []
for acct in ['A', 'B', 'C']:
    for s in STRATEGIES:
        vals = summary[(summary.account==acct) & (summary.strategy==s)].final.values
        med, lo, hi = boot_median_ci(vals)
        rows.append({'account':acct,'strategy':s,'median':med,'ci_lo':lo,'ci_hi':hi})
ci_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
for ax, acct in zip(axes, ['A', 'B', 'C']):
    sub = ci_df[ci_df.account == acct].sort_values('median', ascending=False)
    y = range(len(sub))
    ax.errorbar(sub['median'], y,
                xerr=[sub['median']-sub['ci_lo'], sub['ci_hi']-sub['median']],
                fmt='o', capsize=4)
    ax.set_yticks(list(y))
    ax.set_yticklabels(sub.strategy)
    ax.set_xscale('log')
    ax.axvline(300, color='gray', linestyle=':', label='start $300')
    ax.set_xlabel('Median bankroll across 10 seeds (log $)')
    ax.set_title(f'Account {acct}')
    ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

ci_df.style.format({'median':'${:,.0f}','ci_lo':'${:,.0f}','ci_hi':'${:,.0f}'})

## 6. Bankroll trajectory medians

Per (account, strategy), the median monthly bankroll trajectory across 10 seeds with IQR shading.

In [ ]:
# Reconstruct per-(seed, account, strategy, eval_month) bankrolls.
# Take the last bank_after row per month per (account, strategy, seed).
bets_sorted = bets.sort_values(['account','strategy','seed','eval_month'])
tail = (bets_sorted
        .groupby(['account','strategy','seed','eval_month'])
        .tail(1)[['account','strategy','seed','eval_month','bank_after']])

# Carry $300 start forward where months had zero bets (no bank update)
all_months = sorted(bets.eval_month.unique())
from itertools import product
grid_idx = pd.DataFrame(list(product(
    ['A','B','C'], STRATEGIES, sorted(bets.seed.unique()), all_months
)), columns=['account','strategy','seed','eval_month'])
tail = grid_idx.merge(tail, on=['account','strategy','seed','eval_month'], how='left')
tail = tail.sort_values(['account','strategy','seed','eval_month'])
tail['bank_after'] = tail.groupby(['account','strategy','seed'])['bank_after'].ffill().fillna(300.0)

fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=True)
for ax, acct in zip(axes, ['A','B','C']):
    for s in STRATEGIES:
        sub = tail[(tail.account==acct) & (tail.strategy==s)]
        pv = sub.pivot(index='eval_month', columns='seed', values='bank_after').sort_index()
        med = pv.median(axis=1)
        lo = pv.quantile(0.25, axis=1)
        hi = pv.quantile(0.75, axis=1)
        l, = ax.plot(med.index, med.values, marker='o', label=s)
        ax.fill_between(med.index, lo.values, hi.values, alpha=0.12, color=l.get_color())
    ax.set_yscale('log')
    ax.axhline(300, color='gray', linestyle=':')
    ax.set_ylabel(f'Account {acct}  bankroll (log $)')
    ax.grid(True, which='both', alpha=0.3)
    if acct == 'A':
        ax.legend(loc='upper left', fontsize=8, ncol=2)
axes[-1].set_xlabel('Eval month')
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.suptitle('Median bankroll trajectory across 10 seeds  (IQR shaded)', y=1.00)
fig.tight_layout()
plt.show()

## 7. Account-by-account decision

For each account we list the strategy with the best seed-median bankroll, and whether the median-PnL-diff confidence intervals against alternatives exclude zero.

In [ ]:
for acct in ['A', 'B', 'C']:
    print(f'\n========== Account {acct} ==========')
    sub = ci_df[ci_df.account == acct].sort_values('median', ascending=False)
    top = sub.iloc[0]
    print(f'  Best seed-median strategy: {top.strategy}')
    print(f'    median ${top["median"]:>14,.0f}   95% CI [${top.ci_lo:,.0f}, ${top.ci_hi:,.0f}]')

    # Comparisons of top vs alternatives — median PnL-diff CI from pair_dfs
    others = sub.iloc[1:]
    pair = agg_pair(pair_dfs[acct])
    print(f'  vs alternatives (median across seeds of per-bet PnL diff, 95% CI):')
    for _, r in others.iterrows():
        match = pair[((pair.A == top.strategy) & (pair.B == r.strategy)) |
                     ((pair.B == top.strategy) & (pair.A == r.strategy))]
        if len(match) == 0: continue
        m = match.iloc[0]
        signflip = -1 if m.B == top.strategy else 1
        diff   = signflip * m.median_pnl_diff
        ci_lo  = signflip * (m.median_ci_lo if signflip == 1 else m.median_ci_hi)
        ci_hi  = signflip * (m.median_ci_hi if signflip == 1 else m.median_ci_lo)
        sig = (ci_lo > 0 or ci_hi < 0)
        marker = '★ SIG' if sig else '   ns'
        print(f'    {marker}  {top.strategy} − {r.strategy:18s}  '
              f'median_diff={diff:+.3f}  CI [{ci_lo:+.3f}, {ci_hi:+.3f}]  '
              f'(other median ${r["median"]:,.0f})')

## 8. Final deployment recommendation

_Fill this cell after cells 1-7 have rendered. It should answer:_

1. Which strategy goes on each account, and why.
2. Where significance is genuine vs where the gap is within noise.
3. What can be retired (claims from earlier work that no longer survive).
4. What still needs to be updated in DEPLOY.md and STATUS.md.

## 9. Cross-window robustness check (earlier eval window)

Repeats the same 10-seed × 8-strategy grid on a non-overlapping **earlier** eval window: **2024-07-01 → 2025-06-30** (12 months).

Strategies are shifted back 12 months so the age structure matches:
- `frozen_2022_01`, `frozen_2023_01`, **`frozen_2024_01`** (analog of `frozen_2025_01` — predicted winner under the 6-12 mo age hypothesis), `frozen_2024_07`
- rolling 1/3/6/12mo anchored at 2024-07-01

If the age sweet-spot finding is real (not just a property of the 2025-26 period), the analog winner `frozen_2024_01` should also win this earlier grid.

**Data source:** `data/interim/all_accounts_seed_grid_earlier_bets.parquet`, `artifacts/metrics/all_accounts_seed_grid_earlier.json` (produced by `scripts/all_accounts_seed_grid_earlier.py`).

In [ ]:
summary_e = pd.DataFrame(json.loads((ROOT / 'artifacts/metrics/all_accounts_seed_grid_earlier.json').read_text())['summary'])
bets_e    = pd.read_parquet(ROOT / 'data/interim/all_accounts_seed_grid_earlier_bets.parquet')
STRATEGIES_E = ['frozen_2022_01','frozen_2023_01','frozen_2024_01','frozen_2024_07',
                'rolling_1mo','rolling_3mo','rolling_6mo','rolling_12mo']
print(f'Earlier-window summary rows: {len(summary_e)}   bet rows: {len(bets_e):,}')
print(f'Seeds: {sorted(summary_e.seed.unique())}')

### 9a. Bankroll boxplots — earlier window

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
for ax, acct in zip(axes, ['A','B','C']):
    data = [summary_e[(summary_e.account==acct) & (summary_e.strategy==s)].final.values for s in STRATEGIES_E]
    ax.boxplot(data, labels=STRATEGIES_E, showfliers=True)
    ax.set_yscale('log')
    ax.set_ylabel(f'Account {acct}  final bankroll (log $)')
    ax.axhline(300, color='gray', linestyle=':', linewidth=1)
    ax.grid(True, which='both', alpha=0.3)
axes[-1].set_xlabel('Strategy')
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.suptitle('EARLIER window (2024-07 → 2025-06): bankroll across 10 seeds', y=1.00)
fig.tight_layout()
plt.show()

### 9b. Median bankroll — earlier window summary table

In [ ]:
tab_e = summary_e.groupby(['account','strategy']).apply(fmt_row).reset_index()
tab_e = tab_e.sort_values(['account','median'], ascending=[True, False]).reset_index(drop=True)
tab_e['is_best'] = tab_e['median'] == tab_e.groupby('account')['median'].transform('max')
hilite(tab_e)

### 9c. Cross-window comparison

Side-by-side: for each account, the rank order of strategies in the **earlier** window vs the **main** window. If the 6-12mo age hypothesis is real, the hypothesis-winner should top both columns:
- earlier window predicted winner: `frozen_2024_01` (cutoff is 6 months before EVAL_START=2024-07)
- main window predicted winner: `frozen_2025_01` (cutoff is 6 months before EVAL_START=2025-07)

In [ ]:
# Strategy-name alignment by age role across the two windows
age_role = {
    'too_old_undertrained':    ('frozen_2022_01', 'frozen_2023_01'),
    'old_18_29mo':             ('frozen_2023_01', 'frozen_2024_01'),
    'sweet_spot_6_17mo':       ('frozen_2024_01', 'frozen_2025_01'),  # the predicted winner
    'fresh_0_11mo':            ('frozen_2024_07', 'frozen_2025_07'),
    'rolling_1mo':             ('rolling_1mo', 'rolling_1mo'),
    'rolling_3mo':             ('rolling_3mo', 'rolling_3mo'),
    'rolling_6mo':             ('rolling_6mo', 'rolling_6mo'),
    'rolling_12mo':            ('rolling_12mo', 'rolling_12mo'),
}

rows = []
for acct in ['A','B','C']:
    med_e = summary_e[summary_e.account==acct].groupby('strategy')['final'].median()
    med_m = summary[summary.account==acct].groupby('strategy')['final'].median()
    rank_e = med_e.rank(ascending=False).astype(int)
    rank_m = med_m.rank(ascending=False).astype(int)
    for role, (s_e, s_m) in age_role.items():
        rows.append({
            'account': acct, 'age_role': role,
            'earlier_strategy': s_e, 'earlier_median': med_e.get(s_e, np.nan), 'earlier_rank': rank_e.get(s_e, np.nan),
            'main_strategy': s_m, 'main_median': med_m.get(s_m, np.nan),    'main_rank':    rank_m.get(s_m, np.nan),
        })
cmp = pd.DataFrame(rows)
cmp.style.format({'earlier_median':'${:,.0f}', 'main_median':'${:,.0f}',
                  'earlier_rank':'{:.0f}',   'main_rank':'{:.0f}'})

### 9d. Replication of the 'sweet-spot' rank by account

Did the hypothesis-winner (age 6-17mo frozen) actually rank #1 in both windows? Pulled out for clarity.

In [ ]:
ss = cmp[cmp.age_role == 'sweet_spot_6_17mo'][['account','earlier_strategy','earlier_rank','main_strategy','main_rank']]
ss['both_first'] = (ss.earlier_rank==1) & (ss.main_rank==1)
ss.style.format({'earlier_rank':'{:.0f}','main_rank':'{:.0f}'})

### 9e. Verdict

If `both_first == True` for all three accounts, the 6-12mo age sweet spot replicates strongly across the two windows. If only A and B hit #1 (the real-model accounts), the hypothesis is robust for the real model but the corrupted model is less sensitive — both 'sweet spot' and 'fresh' frozen cutoffs work for it within seed noise.